In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
# !pip install -U \
#     torch==2.3.0+cu121 \
#     torchvision==0.18.0+cu121 \
#     torchaudio==2.3.0+cu121 \
#     bitsandbytes==0.43.3 \
#     triton==2.3.0 \
#     peft==0.10.0 \
#     trl==0.9.6 \
#     transformers==4.37.2 \
#     accelerate==0.27.2 \
#     datasets==2.16.0 \
#     evaluate==0.4.2 \
#     tensorboard==2.20.0 \
#     scipy==1.11.4 \
#     pandas==2.1.4 \
#     tqdm==4.67.1 \
#     rouge-score==0.1.2\
#     nltk==3.8.1\
#     sentencepiece==0.1.99\
#     --extra-index-url https://download.pytorch.org/whl/cu121


In [3]:
# pip install -U evaluate


In [4]:
# ============================================================================
# CELL 1: IMPORTS
# ============================================================================
import os
import torch
import gc
import json
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from evaluate import load
import nltk
nltk.download('punkt', quiet=True)

print("✓ All imports successful!\n")

✓ All imports successful!



In [5]:
# ============================================================================
# CELL 2: CONFIGURATION
# ============================================================================
# Model Configuration
model_name = "google/flan-t5-large"
new_model = "flan-t5-query-augmentation"

# Dataset files (local JSONL files)
dataset_file = "filtered_final_questions.jsonl"  # Your 3000 samples file


In [6]:
# Training Hyperparameters
per_device_train_batch_size = 8
per_device_eval_batch_size = 8
gradient_accumulation_steps = 1
learning_rate = 3e-4
max_grad_norm = 1.0
weight_decay = 0.01
num_train_epochs = 5
max_steps = -1  # -1 means train for num_train_epochs
warmup_ratio = 0.03

# Sequence Settings
max_input_length = 256
max_target_length = 256

In [7]:
# Precision Settings
fp16 = False # Use FP16 for A100
bf16 = True

In [8]:

# Optimizer and Scheduler
optim = "adamw_torch_fused"
lr_scheduler_type = "cosine"

In [9]:

# Logging and Saving
save_steps = 100
logging_steps = 10
output_dir = "flan-t5-query-augmentation"
report_to = "tensorboard"
tb_log_dir = f"{output_dir}/logs"

In [10]:
# Data split ratios
train_split = 0.8
val_split = 0.1
test_split = 0.1


In [11]:
# Device Configuration
device_map = "auto"
seed = 42

In [12]:
print("="*80)
print("CONFIGURATION LOADED")
print("="*80)
print(f"Model: {model_name}")
print(f"Dataset file: {dataset_file}")
print(f"Output dir: {output_dir}")
print(f"Batch size: {per_device_train_batch_size}")
print(f"Learning rate: {learning_rate}")
print(f"Epochs: {num_train_epochs}")
print(f"FP16: {fp16}")
print("="*80 + "\n")

CONFIGURATION LOADED
Model: google/flan-t5-large
Dataset file: filtered_final_questions.jsonl
Output dir: flan-t5-query-augmentation
Batch size: 8
Learning rate: 0.0003
Epochs: 5
FP16: False



In [13]:
# ============================================================================
# CELL 3: LOAD AND PREPARE DATASET
# ============================================================================
print("="*80)
print("LOADING DATASET")
print("="*80)

def load_jsonl(file_path):
    """Load JSONL file"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

LOADING DATASET


In [14]:
# Load dataset
print(f"Loading data from {dataset_file}...")
data = load_jsonl(dataset_file)
dataset = Dataset.from_list(data)

print(f"✓ Loaded {len(dataset)} samples")
print(f"✓ Columns: {dataset.column_names}\n")


Loading data from filtered_final_questions.jsonl...
✓ Loaded 3078 samples
✓ Columns: ['rewritten_question', 'question']



In [15]:
# Split dataset
print("Splitting dataset...")
train_test = dataset.train_test_split(test_size=(1 - train_split), seed=seed)
test_val = train_test['test'].train_test_split(
    test_size=test_split / (val_split + test_split), 
    seed=seed
)

train_dataset = train_test['train']
val_dataset = test_val['train']
test_dataset = test_val['test']

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Validation samples: {len(val_dataset)}")
print(f"✓ Test samples: {len(test_dataset)}\n")

Splitting dataset...
✓ Train samples: 2462
✓ Validation samples: 308
✓ Test samples: 308



In [16]:
# Shuffle datasets
train_dataset = train_dataset.shuffle(seed=seed)
val_dataset = val_dataset.shuffle(seed=seed)

# Print sample
print("="*80)
print("SAMPLE DATA:")
print("="*80)
sample = train_dataset[0]
print(f"Question: {sample['question']}")
print(f"Rewritten: {sample['rewritten_question'][:100]}...")
print("="*80 + "\n")

SAMPLE DATA:
Question: cause of lunar craters
Rewritten: Lunar craters are the result of meteorite impacts....



In [17]:
# ============================================================================
# CELL 4: LOAD MODEL AND TOKENIZER
# ============================================================================
print("="*80)
print("LOADING MODEL AND TOKENIZER")
print("="*80)

# Load tokenizer
print(f"Loading tokenizer: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ Tokenizer loaded")

# Load model
print(f"Loading model: {model_name}...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map=device_map,
    torch_dtype=torch.float16 if fp16 else torch.float32,
)

print("✓ Model loaded successfully!")
print(f"✓ Model parameters: {model.num_parameters() / 1e6:.2f}M")
print(f"✓ Device: {next(model.parameters()).device}\n")

LOADING MODEL AND TOKENIZER
Loading tokenizer: google/flan-t5-large...


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✓ Tokenizer loaded
Loading model: google/flan-t5-large...
✓ Model loaded successfully!
✓ Model parameters: 783.15M
✓ Device: cuda:0



In [18]:
# ============================================================================
# CELL 5: TOKENIZE DATASETS
# ============================================================================
print("="*80)
print("TOKENIZING DATASETS")
print("="*80)

def preprocess_function(examples):
    """Tokenize inputs and outputs"""
    # Add instruction prefix to input
    inputs = [f"Rewrite the following question without changing its meaning. question: \n{q}"  for q in examples['question']]
    targets = examples['rewritten_question']
    
    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding='max_length'
    )
    
    # Tokenize targets
    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


TOKENIZING DATASETS


In [19]:
# Tokenize all datasets
print("Tokenizing training set...")
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

print("Tokenizing validation set...")
tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print("Tokenizing test set...")
tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test"
)

print("✓ All datasets tokenized!\n")

Tokenizing training set...


Tokenizing train:   0%|          | 0/2462 [00:00<?, ? examples/s]

Tokenizing validation set...


Tokenizing validation:   0%|          | 0/308 [00:00<?, ? examples/s]

Tokenizing test set...


Tokenizing test:   0%|          | 0/308 [00:00<?, ? examples/s]

✓ All datasets tokenized!



In [20]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [21]:
# ============================================================================
# CELL 6: EVALUATION METRICS
# ============================================================================
print("="*80)
print("SETTING UP METRICS")
print("="*80)

rouge = load("rouge")

SETTING UP METRICS


In [30]:
import numpy as np  # <-- add this line
def compute_metrics(eval_pred):
    """Compute ROUGE scores"""
    predictions, labels = eval_pred

    # 🩹 Handle model outputs that are tuples (common in seq2seq models)
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # 🩹 Convert logits to token IDs if needed
    if predictions.ndim == 3:  # shape: (batch, seq_len, vocab_size)
        predictions = np.argmax(predictions, axis=-1)

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels with the pad token ID
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # Format results
    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result


print("✓ Metrics configured (ROUGE-1, ROUGE-2, ROUGE-L)\n")


✓ Metrics configured (ROUGE-1, ROUGE-2, ROUGE-L)



In [31]:
# ============================================================================
# CELL 7: TRAINING ARGUMENTS
# ============================================================================
print("="*80)
print("CONFIGURING TRAINING")
print("="*80)

training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    
    # Optimization
    optim=optim,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    max_grad_norm=max_grad_norm,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    
    # Precision
    fp16=fp16,
    bf16=bf16,
    
    # Logging and Saving
    logging_dir=tb_log_dir,
    logging_steps=logging_steps,
    save_steps=save_steps,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    
    # Model Selection
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    greater_is_better=True,
    save_total_limit=2,
    
    # Other
    report_to=report_to,
    dataloader_num_workers=4,
    seed=seed,
)

print("✓ Training arguments configured!")
print(f"  - Total epochs: {num_train_epochs}")
print(f"  - Batch size: {per_device_train_batch_size}")
print(f"  - Learning rate: {learning_rate}")
print(f"  - Optimizer: {optim}")
print(f"  - Scheduler: {lr_scheduler_type}")
print(f"  - FP16: {fp16}")
print(f"  - Save strategy: epoch")
print(f"  - Evaluation strategy: epoch\n")

#+

CONFIGURING TRAINING
✓ Training arguments configured!
  - Total epochs: 5
  - Batch size: 8
  - Learning rate: 0.0003
  - Optimizer: adamw_torch_fused
  - Scheduler: cosine
  - FP16: False
  - Save strategy: epoch
  - Evaluation strategy: epoch



In [32]:
print("="*80)
print("INITIALIZING TRAINER")
print("="*80)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✓ Trainer initialized successfully!\n")

INITIALIZING TRAINER
✓ Trainer initialized successfully!



In [33]:
# CELL 9: START TRAINING
# ============================================================================
print("="*80)
print("STARTING TRAINING")
print("="*80)
print("Training will start now. Monitor progress below...")
print("You can also view TensorBoard logs in real-time:")
print(f"  tensorboard --logdir {tb_log_dir} --host 0.0.0.0")
print("="*80 + "\n")

# Train the model
train_result = trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETED!")
print("="*80)
print(f"✓ Final train loss: {train_result.training_loss:.4f}")
print(f"✓ Training time: {train_result.metrics['train_runtime']:.2f}s")
print("="*80 + "\n")

STARTING TRAINING
Training will start now. Monitor progress below...
You can also view TensorBoard logs in real-time:
  tensorboard --logdir flan-t5-query-augmentation/logs --host 0.0.0.0



Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.019600,0.177737,48.914100,22.501800,47.054600,46.958100
2,0.089600,0.146662,49.548000,22.861700,47.841800,47.826800
3,0.040900,0.173682,49.264600,23.633000,47.747200,47.724900
4,0.020900,0.200874,49.720600,23.888800,48.249200,48.144800
5,0.014900,0.220147,49.662500,23.826500,48.014300,47.947900


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



TRAINING COMPLETED!
✓ Final train loss: 0.0373
✓ Training time: 641.62s



In [34]:
# ============================================================================
# CELL 10: SAVE MODEL
# ============================================================================
print("="*80)
print("SAVING MODEL")
print("="*80)

# Save final model
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ Model saved to: {output_dir}")
print(f"✓ Tokenizer saved to: {output_dir}\n")

SAVING MODEL
✓ Model saved to: flan-t5-query-augmentation
✓ Tokenizer saved to: flan-t5-query-augmentation



In [36]:
#Save training metrics
with open(f"{output_dir}/training_metrics.json", 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

print(f"✓ Training metrics saved to: {output_dir}/training_metrics.json\n")


✓ Training metrics saved to: flan-t5-query-augmentation/training_metrics.json



In [37]:
#============================================================================
# CELL 11: EVALUATE ON TEST SET
# ============================================================================
print("="*80)
print("FINAL EVALUATION ON TEST SET")
print("="*80)

test_results = trainer.evaluate(tokenized_test)

print("\nTest Set Results:")
print("-" * 80)
for key, value in test_results.items():
    print(f"  {key}: {value}")
print("="*80 + "\n")


FINAL EVALUATION ON TEST SET



Test Set Results:
--------------------------------------------------------------------------------
  eval_loss: 0.21259315311908722
  eval_rouge1: 46.6866
  eval_rouge2: 21.257
  eval_rougeL: 44.8893
  eval_rougeLsum: 44.9643
  eval_runtime: 16.3592
  eval_samples_per_second: 18.827
  eval_steps_per_second: 2.384
  epoch: 5.0



In [38]:

# Save test results
with open(f"{output_dir}/test_results.json", 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"✓ Test results saved to: {output_dir}/test_results.json\n")


✓ Test results saved to: flan-t5-query-augmentation/test_results.json



In [39]:
# ============================================================================
# CELL 12: TEST INFERENCE
# ============================================================================
print("="*80)
print("TESTING INFERENCE")
print("="*80)

test_examples = [
    "Earth science is the study of",
    "What causes earthquakes?",
    "How do volcanoes form?",
    "What is photosynthesis?",
    "Why is the sky blue?"
]

print("\nSample predictions:\n")

for example in test_examples:
    input_text = f"Rewrite this question: {example}"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_length=max_target_length,
        num_beams=4,
        early_stopping=True,
        temperature=0.7,
    )
    
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Original:  {example}")
    print(f"Rewritten: {prediction}")
    print("-" * 80)

TESTING INFERENCE

Sample predictions:



/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Original:  Earth science is the study of
Rewritten: The scientific investigation of our planet encompasses the examination of its physical processes and natural features.
--------------------------------------------------------------------------------
Original:  What causes earthquakes?
Rewritten: Earth's crust is the primary cause of earthquakes.
--------------------------------------------------------------------------------
Original:  How do volcanoes form?
Rewritten: What processes lead to the formation of volcanic features?
--------------------------------------------------------------------------------
Original:  What is photosynthesis?
Rewritten: What process converts sunlight into energy?
--------------------------------------------------------------------------------
Original:  Why is the sky blue?
Rewritten: What color does the sky exhibit when illuminated by sunlight?
--------------------------------------------------------------------------------


In [40]:

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)
print(f"\nYour fine-tuned model is ready!")
print(f"Model location: {output_dir}")
print(f"\nTo use the model later:")
print(f"  from transformers import AutoTokenizer, AutoModelForSeq2SeqLM")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{output_dir}')")
print(f"  model = AutoModelForSeq2SeqLM.from_pretrained('{output_dir}')")
print("="*80)


ALL DONE!

Your fine-tuned model is ready!
Model location: flan-t5-query-augmentation

To use the model later:
  from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
  tokenizer = AutoTokenizer.from_pretrained('flan-t5-query-augmentation')
  model = AutoModelForSeq2SeqLM.from_pretrained('flan-t5-query-augmentation')


In [41]:
def inference(question, model_path=output_dir, device='cuda'):
    """
    Use the trained model for inference
    
    Args:
        question: Input question to rewrite
        model_path: Path to saved model
        device: 'cuda' or 'cpu'
    
    Returns:
        Rewritten question
    """
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
    
    input_text = f"Rewrite this question: {question}"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    
    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        early_stopping=True,
        temperature=0.7,
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage:
# rewritten = inference("What is machine learning?")
# print(rewritten)

print("\n✓ Inference function defined and ready to use")


✓ Inference function defined and ready to use
